In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dask.dataframe as dd
import trenchripper as tr
import pandas as pd

%matplotlib inline

In [9]:
# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

dask_wd = "/home/lag36/scratch/lag36/dask"

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="3:00:00",
    local=False,
    n_workers=20,
    n_workers_min=20,#200,
    memory="8GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

# dask_controller = tr.trcluster.dask_controller(
#     walltime="2:00:00",
#     local=False,
#     n_workers=50,
#     n_workers_min=50,
#     memory="8GB",
#     working_directory=dask_wd,
# )
# dask_controller.startdask()

In [ ]:
dask_controller.daskclient

In [ ]:
import pandas as pd
df_2_1 = pd.read_hdf('/home/lag36/scratch/lag36/2024-10-31_lLAG8_MBM_Run-2/Barcodes/barcode_output_df_fiducials_hamming-2_rm-none.hdf5')
df_2_1

In [13]:
# Translates from dd to df
bc_kymo_metadata = dd.read_parquet("/home/lag36/scratch/lag36/2024-10-31_lLAG8_MBM_Run-2/Barcodes/kymograph/metadata", calculate_divisions=True)
bc_kymo_metadata_df = bc_kymo_metadata.compute()
# Change the type of the lane orientation column to category
bc_kymo_metadata_df["lane orientation"] = bc_kymo_metadata_df["lane orientation"].astype(object)
bc_kymo_metadata_df.to_hdf("/home/lag36/scratch/lag36/2024-10-31_lLAG8_MBM_Run-2/Barcodes/kymograph_metadata_df.hdf5", key='data')

In [3]:
bc_kymo_metadata_df = pd.read_hdf("/home/lag36/scratch/lag36/2024-10-31_lLAG8_MBM_Run-2/Barcodes/kymograph_metadata_df.hdf5", key='data')

In [ ]:
df_2_1

In [ ]:
df_2_1.columns

In [ ]:
bc_kymo_metadata_df.columns

In [6]:
df_lookup = (df_2_1
             .loc[:,['trenchid', 'Barcode Signal', 'Barcode', 'closest_match',  'barcode depth', 'Closest Hamming Distance', 'Category', 'barcode', 'barcodeid']]
             .merge(bc_kymo_metadata_df.groupby('trenchid').first().reset_index(),
                    left_on='trenchid',
                    right_on='trenchid', how='left')
                )

In [24]:
### QC segmentation
import h5py
import matplotlib.pyplot as plt
import numpy as np

def show_kymograph(
        path_prefix,
        path_kymo_dir_prefix,
        channels,
        cmap,
        kymo_file_index,
        trench_index,
        t_indices,
        figsize,
        Barcode = None,
        barcode_phe = None #translated from seq barcode
        ):
    
    n_cycles_per_ch = 15 # TODO GENERALIZE
    with h5py.File(path_prefix + path_kymo_dir_prefix + str(kymo_file_index) + ".hdf5", "r") as infile:
        for i_ch, ch in enumerate(channels):
            img = infile[ch][trench_index][t_indices[0]]
            for ind in t_indices[1:]:
                img_ind = infile[ch][trench_index][ind]
                img = np.concatenate((img, img_ind), axis=1)
            plt.figure(figsize=figsize)
            # Do not show axes ticks
            plt.xticks([])
            plt.yticks([])
            plt.imshow(img, cmap=cmap, aspect=1)
            # Annotate the kymo_file_index and the trench index at the bottom left corner
            plt.annotate(f'kymo_file_index: {kymo_file_index}, trench index: {trench_index}', (10, 15), color='white')
            # Annotate the cycle number at the bottom of each slice
            for i, ind in enumerate(t_indices):
                plt.annotate(f'{ind}', (i*9+9/4, 160), color='white', fontsize=8) # 13 is the width of a slice
            
            # Annotate the Barcode bit at the bottom of each slice
            if Barcode is not None:
                
                for i, bit in enumerate(Barcode[i_ch*n_cycles_per_ch:(i_ch+1)*n_cycles_per_ch]):
                    plt.annotate(f'{bit}', (i*9+9/3, 140), color='yellow', fontsize=8) # 13 is the width of a slice

            # Annotate the barcode bit at the bottom of each slice
            if barcode_phe is not None:
                for i, bit in enumerate(barcode_phe[i_ch*n_cycles_per_ch:(i_ch+1)*n_cycles_per_ch]):
                    plt.annotate(f'{bit}', (i*9+9/3, 120), color='red', fontsize=8) # 13 is the width of a slice

            plt.show()
            plt.close()

def map_bc_to_pheno(barcode,
                    cycleorder = [0,1,6,7,12,13,18,19,24,25,4,5,10,11,16,
                                  2,3,8,9,14,15,20,21,26,27,17,22,23,28,29],
                    ):
    barcode_phe = [-1]*30
    for i, bit in enumerate(barcode):
        # print(i, bit)
        # Find the position of i in cycleorder
        i_phe = cycleorder.index(i)
        barcode_phe[i_phe] = bit
    barcode_phe = ''.join(barcode_phe)
    return barcode_phe

In [8]:
a = df_lookup.loc[df_lookup['Category'] == 'fiducial', 'Barcode'].value_counts()

In [ ]:
a

In [ ]:
a[a == 4]

In [ ]:
df_lookup[df_lookup['closest_match'] <= 10]

In [ ]:
df_lookup.loc[df_lookup['Barcode'] == '111111111111111111111111101110']

In [ ]:
df_lookup.loc[df_lookup['Barcode'] == '111111111111111111111111101110', 'Category']

In [ ]:
trenchid = 107150
bc_kymo_file_index, bc_trench_index, Barcode, barcode = df_lookup.loc[df_lookup['trenchid'] == trenchid, ['File Index', 'File Trench Index', 'Barcode', 'barcode']].values[0]
bc_kymo_file_index = int(bc_kymo_file_index)
bc_trench_index = int(bc_trench_index)
# barcode (lowercase) comes from sequencing (red)
# Barcode (uppercase) comes from trench signal (yellow)
# Barcode, barcode = df_lookup.loc[row, ['Barcode', 'barcode']]
barcode_phe = map_bc_to_pheno(barcode)
figsize = (4, 8)
path_prefix = "/home/lag36/scratch/lag36/2024-10-31_lLAG8_MBM_Run-2/Barcodes/"
path_kymo_dir_prefix = "kymograph/kymograph_"
# channels = ["BF", "RFP", "Cy5"]
channels = ["RFP", "Cy5"]
cmap = "gray"
t_indices = np.arange(0,15)
show_kymograph(path_prefix, path_kymo_dir_prefix, channels, cmap, bc_kymo_file_index, bc_trench_index, t_indices, figsize, Barcode, barcode_phe)
show_kymograph(path_prefix, path_kymo_dir_prefix, ['BF'], cmap, bc_kymo_file_index, bc_trench_index, t_indices, figsize, None, None)